# Behavior Feature Combination Experiment

이 노트북은 최소한의 student behavior feature 3개를 조합하면서 OULAD at-risk 예측 실험을 진행한다.

사용 feature는 정확히 3개다.

| alias | raw column | 의미 |
| --- | --- | --- |
| `asm_days_early_mean` | `avg_days_before_due_until_cutoff` | 제출일이 정시 대비 평균적으로 얼마나 이른지 |
| `asm_count` | `assessment_count_until_cutoff` | cutoff까지 관측된 과제/평가 제출 수 |
| `vle_last_day` | `last_activity_date_until_cutoff` | cutoff까지 관측된 마지막 VLE 접속/활동일 |

실험은 두 단계로 진행한다.

1. 행동 지표 3개에 대해 `3C1`, `3C2`, `3C3` 조합을 모두 실험한다.
2. 같은 조합에 base feature인 `vle_click_total`, `vle_active_days`를 추가해 동일하게 실험한다.

즉 feature set은 행동-only 7개와 base+행동 7개, 총 14개다.

## 1. Setup

프로젝트 루트를 자동으로 찾고, 실험 스크립트의 공통 함수와 모델 정의를 불러온다. 원본 raw data는 수정하지 않는다.

In [6]:
from pathlib import Path
import importlib
import sys
import warnings

import pandas as pd
from sklearn.model_selection import ParameterGrid, StratifiedKFold

warnings.filterwarnings("ignore", category=UserWarning)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "run_model_experiments.py").exists():
            return candidate
    raise FileNotFoundError("Could not find project root")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import run_model_experiments as exp
exp = importlib.reload(exp)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULT_DIR = PROJECT_ROOT / "data" / "report_tables" / "model_results"
ALL_RESULTS_PATH = RESULT_DIR / "feature_engineering_grid_results.csv"
BEST_RESULTS_PATH = RESULT_DIR / "feature_engineering_best_results.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULT_DIR:", RESULT_DIR.relative_to(PROJECT_ROOT))


PROJECT_ROOT: C:\Users\osca0\Dev\studyDataMining\teamproject
RESULT_DIR: data\report_tables\model_results


## 2. Experiment Configuration

이번 notebook은 행동 feature 조합 7개와 base+행동 feature 조합 7개를 사용한다. cutoff는 week 5, week 7, week 10을 모두 비교한다.

행동 feature 3개는 `asm_days_early_mean`, `asm_count`, `vle_last_day`이다. base feature 2개는 `vle_click_total`, `vle_active_days`이다.

`RUN_EXPERIMENTS = False` 상태에서는 계획표와 기존 결과만 확인한다. 실제 모델링을 실행하려면 `True`로 바꾼다.

`SKIP_COMPLETED = True`이면 이미 저장된 `cutoff × feature_set × model` 조합은 다시 실행하지 않는다.

In [12]:
CUT_OFFS = ["week5", "week7", "week10"]

BEHAVIOR_ONLY_FEATURE_SETS = [
    "beh_early",  # 3C1: submission earliness
    "beh_count",  # 3C1: assessment submission count
    "beh_last",  # 3C1: last VLE activity day
    "beh_early_count",  # 3C2: earliness + count
    "beh_early_last",  # 3C2: earliness + last VLE day
    "beh_count_last",  # 3C2: count + last VLE day
    "beh_all3",  # 3C3: all behavior features
]

BASE_PLUS_BEHAVIOR_FEATURE_SETS = [
    "base_beh_early",  # base + 3C1: submission earliness
    "base_beh_count",  # base + 3C1: assessment submission count
    "base_beh_last",  # base + 3C1: last VLE activity day
    "base_beh_early_count",  # base + 3C2: earliness + count
    "base_beh_early_last",  # base + 3C2: earliness + last VLE day
    "base_beh_count_last",  # base + 3C2: count + last VLE day
    "base_beh_all3",  # base + 3C3: all behavior features
]

FEATURE_SET_NAMES = BEHAVIOR_ONLY_FEATURE_SETS + BASE_PLUS_BEHAVIOR_FEATURE_SETS

MODEL_NAMES = [
    "GaussianNB",
    "LogisticRegression",
    "DecisionTreeClassifier",
    "RandomForestClassifier",
    "KNeighborsClassifier",
    "SVC",
]

# 실제 GridSearchCV를 실행하려면 True로 바꾼다.
RUN_EXPERIMENTS = True #False
SKIP_COMPLETED = True
N_SPLITS = 5
N_JOBS = -1
TARGET_COL = "target_at_risk"


## 3. Feature Combination Definition Check

아래 셀은 14개 feature set이 실제로 어떤 feature를 포함하는지 확인한다.

- `beh_*`: 행동 지표 3개에서 1개, 2개, 3개를 선택한 조합
- `base_beh_*`: 같은 행동 조합에 base feature 2개를 추가한 조합

base feature는 `total_click_until_cutoff_mean_strategy`, `active_days_until_cutoff`이다.

In [4]:
sample_df = pd.read_csv(DATA_DIR / "features_week5_baseline.csv", nrows=100)
feature_sets = exp.build_feature_sets(sample_df)
missing_feature_sets = [name for name in FEATURE_SET_NAMES if name not in feature_sets]
if missing_feature_sets:
    raise KeyError(
        "Feature sets are missing from run_model_experiments.py: "
        f"{missing_feature_sets}. Re-run the Setup cell or restart the notebook kernel."
    )

feature_set_rows = []
for feature_set_name in FEATURE_SET_NAMES:
    features = feature_sets[feature_set_name]
    feature_set_rows.append(
        {
            "feature_set_raw": feature_set_name,
            "feature_set_alias": exp.alias_feature_set_name(feature_set_name),
            "description": exp.FEATURE_SET_DESCRIPTION[feature_set_name],
            "feature_count": len(features),
            "features_alias": ";".join(exp.alias_feature_list(features)),
            "features_raw": ";".join(features),
        }
    )

feature_set_table = pd.DataFrame(feature_set_rows)
feature_set_table


,feature_set_raw,feature_set_alias,description,feature_count,features_alias,features_raw
0,beh_early,beh_early,Behavior 3C1: submission earliness only.,1,asm_days_early_mean,avg_days_before_due_until_cutoff
1,beh_count,beh_count,Behavior 3C1: assessment submission count only.,1,asm_count,assessment_count_until_cutoff
2,beh_last,beh_last,Behavior 3C1: last VLE activity day only.,1,vle_last_day,last_activity_date_until_cutoff
3,beh_early_count,beh_early_count,Behavior 3C2: submission earliness plus assess...,2,asm_days_early_mean;asm_count,avg_days_before_due_until_cutoff;assessment_co...
4,beh_early_last,beh_early_last,Behavior 3C2: submission earliness plus last V...,2,asm_days_early_mean;vle_last_day,avg_days_before_due_until_cutoff;last_activity...
5,beh_count_last,beh_count_last,Behavior 3C2: assessment submission count plus...,2,asm_count;vle_last_day,assessment_count_until_cutoff;last_activity_da...
6,beh_all3,beh_all3,Behavior 3C3: all three minimal behavior featu...,3,asm_days_early_mean;asm_count;vle_last_day,avg_days_before_due_until_cutoff;assessment_co...
7,base_beh_early,base_beh_early,Base features plus behavior 3C1: submission ea...,3,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...
8,base_beh_count,base_beh_count,Base features plus behavior 3C1: assessment su...,3,vle_click_total;vle_active_days;asm_count,total_click_until_cutoff_mean_strategy;active_...
9,base_beh_last,base_beh_last,Base features plus behavior 3C1: last VLE acti...,3,vle_click_total;vle_active_days;vle_last_day,total_click_until_cutoff_mean_strategy;active_...


## 4. Input Validation Summary

모델링 전에 각 cutoff 파일의 행 수, 양성 비율, feature set별 결측 여부를 확인한다. 선택된 숫자 feature에 결측이 남아 있으면 실험을 중단해야 한다.

In [5]:
validation_rows = []

for cutoff in CUT_OFFS:
    path = DATA_DIR / f"features_{cutoff}_baseline.csv"
    df = pd.read_csv(path)
    cutoff_feature_sets = exp.build_feature_sets(df)
    if TARGET_COL not in df.columns:
        raise ValueError(f"{cutoff} missing target column: {TARGET_COL}")

    for feature_set_name in FEATURE_SET_NAMES:
        selected_features = cutoff_feature_sets[feature_set_name]
        missing_features = [col for col in selected_features if col not in df.columns]
        if missing_features:
            raise ValueError(f"{cutoff}/{feature_set_name} missing features: {missing_features}")
        if "date_unregistration" in selected_features or "id_student" in selected_features:
            raise ValueError(f"Leakage/key feature selected unexpectedly: {feature_set_name}")

        validation_rows.append(
            {
                "cutoff": cutoff,
                "feature_set_alias": exp.alias_feature_set_name(feature_set_name),
                "rows": len(df),
                "positive_rate": df[TARGET_COL].mean(),
                "missing_cells_in_selected_features": int(df[selected_features].isna().sum().sum()),
                "feature_count": len(selected_features),
            }
        )

validation_table = pd.DataFrame(validation_rows)
validation_table


,cutoff,feature_set_alias,rows,positive_rate,missing_cells_in_selected_features,feature_count
0,week5,beh_early,27229,0.435161,0,1
1,week5,beh_count,27229,0.435161,0,1
2,week5,beh_last,27229,0.435161,0,1
3,week5,beh_early_count,27229,0.435161,0,2
4,week5,beh_early_last,27229,0.435161,0,2
5,week5,beh_count_last,27229,0.435161,0,2
6,week5,beh_all3,27229,0.435161,0,3
7,week5,base_beh_early,27229,0.435161,0,3
8,week5,base_beh_count,27229,0.435161,0,3
9,week5,base_beh_last,27229,0.435161,0,3


## 5. Experiment Plan

각 row는 하나의 GridSearchCV 실행 단위다.

```text
cutoff × feature_combination × model
```

`param_candidates`는 해당 모델에서 평가할 parameter 조합 수다.

In [6]:
model_specs = exp.build_model_specs()


def count_param_candidates(param_grid) -> int:
    return len(list(ParameterGrid(param_grid)))


plan_rows = []
for cutoff in CUT_OFFS:
    cutoff_df = pd.read_csv(DATA_DIR / f"features_{cutoff}_baseline.csv", nrows=100)
    cutoff_feature_sets = exp.build_feature_sets(cutoff_df)
    missing_feature_sets = [name for name in FEATURE_SET_NAMES if name not in cutoff_feature_sets]
    if missing_feature_sets:
        raise KeyError(
            f"{cutoff} missing feature set definitions: {missing_feature_sets}. "
            "Re-run the Setup cell or restart the notebook kernel."
        )
    for feature_set_name in FEATURE_SET_NAMES:
        selected_features = cutoff_feature_sets[feature_set_name]
        for model_name in MODEL_NAMES:
            plan_rows.append(
                {
                    "cutoff": cutoff,
                    "feature_set_raw": feature_set_name,
                    "feature_set_alias": exp.alias_feature_set_name(feature_set_name),
                    "model": model_name,
                    "feature_count": len(selected_features),
                    "param_candidates": count_param_candidates(model_specs[model_name]["param_grid"]),
                    "features_alias": ";".join(exp.alias_feature_list(selected_features)),
                    "features_raw": ";".join(selected_features),
                }
            )

experiment_plan = pd.DataFrame(plan_rows)
print("GridSearchCV executions:", len(experiment_plan))
print("Expected all_results rows:", int(experiment_plan["param_candidates"].sum()))
experiment_plan


GridSearchCV executions: 252
Expected all_results rows: 13776


,cutoff,feature_set_raw,feature_set_alias,model,feature_count,param_candidates,features_alias,features_raw
0,week5,beh_early,beh_early,GaussianNB,1,4,asm_days_early_mean,avg_days_before_due_until_cutoff
1,week5,beh_early,beh_early,LogisticRegression,1,24,asm_days_early_mean,avg_days_before_due_until_cutoff
2,week5,beh_early,beh_early,DecisionTreeClassifier,1,216,asm_days_early_mean,avg_days_before_due_until_cutoff
3,week5,beh_early,beh_early,RandomForestClassifier,1,48,asm_days_early_mean,avg_days_before_due_until_cutoff
4,week5,beh_early,beh_early,KNeighborsClassifier,1,12,asm_days_early_mean,avg_days_before_due_until_cutoff
...,...,...,...,...,...,...,...,...
247,week10,base_beh_all3,base_beh_all3,LogisticRegression,5,24,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...
248,week10,base_beh_all3,base_beh_all3,DecisionTreeClassifier,5,216,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...
249,week10,base_beh_all3,base_beh_all3,RandomForestClassifier,5,48,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...
250,week10,base_beh_all3,base_beh_all3,KNeighborsClassifier,5,12,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...


## 6. Completed Combination Check

기존 `best_results`에 같은 조합이 있으면 completed로 표시한다. `SKIP_COMPLETED = True`일 때 completed 조합은 모델링 셀에서 건너뛴다.

In [7]:
def load_existing_best_results() -> pd.DataFrame:
    if not BEST_RESULTS_PATH.exists():
        return pd.DataFrame()
    return exp.add_alias_columns_to_results(pd.read_csv(BEST_RESULTS_PATH))


existing_best = load_existing_best_results()
completed_keys = set()
if not existing_best.empty:
    completed_keys = set(
        map(tuple, existing_best[["cutoff", "feature_set_raw", "model"]].drop_duplicates().values.tolist())
    )

experiment_plan["completed"] = experiment_plan.apply(
    lambda row: (row["cutoff"], row["feature_set_raw"], row["model"]) in completed_keys,
    axis=1,
)

print("Completed combinations:", int(experiment_plan["completed"].sum()))
print("Remaining combinations:", int((~experiment_plan["completed"]).sum()))
experiment_plan


Completed combinations: 0
Remaining combinations: 252


,cutoff,feature_set_raw,feature_set_alias,model,feature_count,param_candidates,features_alias,features_raw,completed
0,week5,beh_early,beh_early,GaussianNB,1,4,asm_days_early_mean,avg_days_before_due_until_cutoff,False
1,week5,beh_early,beh_early,LogisticRegression,1,24,asm_days_early_mean,avg_days_before_due_until_cutoff,False
2,week5,beh_early,beh_early,DecisionTreeClassifier,1,216,asm_days_early_mean,avg_days_before_due_until_cutoff,False
3,week5,beh_early,beh_early,RandomForestClassifier,1,48,asm_days_early_mean,avg_days_before_due_until_cutoff,False
4,week5,beh_early,beh_early,KNeighborsClassifier,1,12,asm_days_early_mean,avg_days_before_due_until_cutoff,False
...,...,...,...,...,...,...,...,...,...
247,week10,base_beh_all3,base_beh_all3,LogisticRegression,5,24,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...,False
248,week10,base_beh_all3,base_beh_all3,DecisionTreeClassifier,5,216,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...,False
249,week10,base_beh_all3,base_beh_all3,RandomForestClassifier,5,48,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...,False
250,week10,base_beh_all3,base_beh_all3,KNeighborsClassifier,5,12,vle_click_total;vle_active_days;asm_days_early...,total_click_until_cutoff_mean_strategy;active_...,False


## 7. Run Modeling

실제 모델링 셀이다. `RUN_EXPERIMENTS = True`로 바꾼 뒤 실행한다.

각 조합이 끝날 때마다 결과가 CSV에 저장되므로, 중간에 멈춰도 완료된 결과는 남는다.

In [8]:
if not RUN_EXPERIMENTS:
    print("RUN_EXPERIMENTS is False. Set it to True to run GridSearchCV experiments.")
else:
    exp.ensure_dirs()
    exp.write_report()
    scoring = exp.build_scoring()
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=exp.RANDOM_STATE)
    run_id = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

    for cutoff in CUT_OFFS:
        df = exp.read_feature_frame(cutoff)
        cutoff_feature_sets = exp.build_feature_sets(df)

        for feature_set_name in FEATURE_SET_NAMES:
            cutoff_features = cutoff_feature_sets[feature_set_name]
            for model_name in MODEL_NAMES:
                key = (cutoff, feature_set_name, model_name)
                if SKIP_COMPLETED and key in completed_keys:
                    print("Skip completed:", key)
                    continue

                exp.run_one_experiment(
                    df=df,
                    model_specs=model_specs,
                    scoring=scoring,
                    cv=cv,
                    run_id=run_id,
                    cutoff=cutoff,
                    strategy=exp.STRATEGY,
                    feature_set_name=feature_set_name,
                    selected_features=cutoff_features,
                    model_name=model_name,
                    target_col=TARGET_COL,
                    n_jobs=N_JOBS,
                )

    exp.refresh_result_aliases()
    print("Experiment loop complete.")


Wrote reports\model_experiment_logging_report.md
Loaded data\processed\features_week5_baseline.csv shape=(27229, 66)
Running: cutoff=week5, feature_set=beh_early (beh_early), model=GaussianNB, n_rows=27229, positive_rate=0.4352, n_features=1


C:\Users\osca0\AppData\Local\Temp\ipykernel_12080\3589495019.py:8: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  run_id = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.601407 params={"model__var_smoothing": 1e-12}
Running: cutoff=week5, feature_set=beh_early (beh_early), model=LogisticRegression, n_rows=27229, positive_rate=0.4352, n_features=1


C:\Users\osca0\Dev\studyDataMining\teamproject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.601728 params={"model__C": 0.01, "model__class_weight": "balanced", "model__penalty": "l1", "model__solver": "liblinear"}
Running: cutoff=week5, feature_set=beh_early (beh_early), model=DecisionTreeClassifier, n_rows=27229, positive_rate=0.4352, n_features=1
Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.605557 params={"model__ccp_alpha": 0.001, "model__criterion": "gini", "model__max_depth": 5, "model__min_samples_leaf": 1, "model__min_samples_split": 2}
Running: cutoff=week5, feature_set=beh_early (beh_early), model=RandomForestClassifier, n_rows=27229, positive_rate=0.4352, n_features=1
Saved all grid results to: data\report_tables\mo

...

Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.592795 params={"model__C": 0.01, "model__class_weight": "balanced", "model__penalty": "l2", "model__solver": "liblinear"}
Running: cutoff=week10, feature_set=base_beh_early_last (base_beh_early_last), model=DecisionTreeClassifier, n_rows=26062, positive_rate=0.4098, n_features=4
Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.611901 params={"model__ccp_alpha": 0.0, "model__criterion": "gini", "model__max_depth": 3, "model__min_samples_leaf": 1, "model__min_samples_split": 2}
Running: cutoff=week10, feature_set=base_beh_early_last (base_beh_early_last), model=RandomForestClassifier, n_rows=26062, positive_rate=0.4098, n_features=4
Saved al

C:\Users\osca0\Dev\studyDataMining\teamproject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.593407 params={"model__C": 0.01, "model__class_weight": "balanced", "model__penalty": "l2", "model__solver": "liblinear"}
Running: cutoff=week10, feature_set=base_beh_count_last (base_beh_count_last), model=DecisionTreeClassifier, n_rows=26062, positive_rate=0.4098, n_features=4
Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.546602 params={"model__ccp_alpha": 0.0, "model__criterion": "entropy", "model__max_depth": null, "model__min_samples_leaf": 5, "model__min_samples_split": 30}
Running: cutoff=week10, feature_set=base_beh_count_last (base_beh_count_last), model=RandomForestClassifier, n_rows=26062, positive_rate=0.4098, n_features=4
S

C:\Users\osca0\Dev\studyDataMining\teamproject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.593869 params={"model__C": 0.01, "model__class_weight": "balanced", "model__penalty": "l2", "model__solver": "lbfgs"}
Running: cutoff=week10, feature_set=base_beh_all3 (base_beh_all3), model=DecisionTreeClassifier, n_rows=26062, positive_rate=0.4098, n_features=5
Saved all grid results to: data\report_tables\model_results\feature_engineering_grid_results.csv
Saved best results to: data\report_tables\model_results\feature_engineering_best_results.csv
Best F1: 0.591599 params={"model__ccp_alpha": 0.0, "model__criterion": "entropy", "model__max_depth": 10, "model__min_samples_leaf": 5, "model__min_samples_split": 30}
Running: cutoff=week10, feature_set=base_beh_all3 (base_beh_all3), model=RandomForestClassifier, n_rows=26062, positive_rate=0.4098, n_features=5
Saved all grid results to: data

## 8. Load Results

`best_results`는 각 cutoff × model에서 F1 기준 best parameter만 남긴 표다. 분석은 이 표를 중심으로 진행한다.

In [9]:
if ALL_RESULTS_PATH.exists():
    grid_results = exp.add_alias_columns_to_results(pd.read_csv(ALL_RESULTS_PATH))
else:
    grid_results = pd.DataFrame()

if BEST_RESULTS_PATH.exists():
    best_results = exp.add_alias_columns_to_results(pd.read_csv(BEST_RESULTS_PATH))
else:
    best_results = pd.DataFrame()

selected_best = best_results[best_results.get("feature_set_raw", pd.Series(dtype=str)).isin(FEATURE_SET_NAMES)].copy()

print("grid_results shape:", grid_results.shape)
print("best_results shape:", best_results.shape)
print("selected_best shape:", selected_best.shape)


grid_results shape: (13784, 32)
best_results shape: (254, 31)
selected_best shape: (252, 31)


## 9. Evaluation Table

F1 기준으로 정렬해 모델별 성능을 확인한다. early warning 목적이므로 F1과 함께 recall, precision, PR-AUC를 같이 본다.

In [10]:
display_cols = [
    "cutoff",
    "feature_set_alias",
    "model",
    "f1_mean",
    "recall_mean",
    "precision_mean",
    "roc_auc_mean",
    "best_params",
]

if selected_best.empty:
    print("No selected feature-combination results yet. Run the modeling cell first.")
else:
    selected_eval = selected_best[display_cols].sort_values(
        ["cutoff", "feature_set_alias", "f1_mean"], ascending=[True, True, False]
    ).reset_index(drop=True)
    selected_eval


## 10. Best Combination by Cutoff

각 cutoff에서 F1이 가장 높은 feature 조합과 모델을 확인한다.

In [11]:
if not selected_best.empty:
    best_by_cutoff = (
        selected_best.sort_values(["cutoff", "f1_mean"], ascending=[True, False])
        .groupby("cutoff", as_index=False)
        .head(1)
    )
    best_by_cutoff[display_cols]


## 11. Base Feature Added Effect

`beh_*` 조합에 base feature 2개를 추가했을 때 F1이 얼마나 달라지는지 확인한다.

In [12]:
base_pair_map = {
    "base_beh_early": "beh_early",
    "base_beh_count": "beh_count",
    "base_beh_last": "beh_last",
    "base_beh_early_count": "beh_early_count",
    "base_beh_early_last": "beh_early_last",
    "base_beh_count_last": "beh_count_last",
    "base_beh_all3": "beh_all3",
}

if not selected_best.empty:
    behavior_only = selected_best[selected_best["feature_set_raw"].isin(base_pair_map.values())][[
        "cutoff", "model", "feature_set_raw", "f1_mean"
    ]].rename(columns={"feature_set_raw": "behavior_set", "f1_mean": "behavior_f1"})

    base_plus = selected_best[selected_best["feature_set_raw"].isin(base_pair_map.keys())][[
        "cutoff", "model", "feature_set_raw", "feature_set_alias", "f1_mean"
    ]].copy()
    base_plus["behavior_set"] = base_plus["feature_set_raw"].map(base_pair_map)

    base_effect = base_plus.merge(behavior_only, on=["cutoff", "model", "behavior_set"], how="left")
    base_effect["f1_delta_from_base_features"] = base_effect["f1_mean"] - base_effect["behavior_f1"]
    base_effect.sort_values(["cutoff", "f1_delta_from_base_features"], ascending=[True, False])[[
        "cutoff", "model", "behavior_set", "feature_set_alias", "behavior_f1", "f1_mean", "f1_delta_from_base_features"
    ]]


In [30]:
df = pd.read_csv(BEST_RESULTS_PATH)
df2 = pd.read_csv(ALL_RESULTS_PATH)
print(df.columns)
shared_columns = ["cutoff","feature_set","n_features","model",'precision_mean', 'precision_std', 'recall_mean', 'recall_std','f1_mean', 'f1_std', 'roc_auc_mean', 'roc_auc_std']
result_df = df[shared_columns]
result_df = result_df.rename(columns={
    "cutoff": "cutoff_week",
    "precision_mean": "precision_mean_T",
    "recall_mean": "recall_mean_T",
    "f1_mean_score": "f1_mean_T",
    "roc_auc_mean": "roc_auc_mean_T",
    "precision_std": "precision_std_T",
    "recall_std": "recall_std_T",
    "f1_std_score": "f1_std_T",
    "roc_auc_std": "roc_auc_std_T",
})
result_df.to_csv(RESULT_DIR / "feature_engineering_behaviour.csv", index=False)

Index(['run_id', 'created_at_utc', 'cutoff', 'strategy', 'feature_set',
       'model', 'n_rows', 'positive_class', 'positive_rate', 'target',
       'n_features', 'features', 'best_params', 'selected_by',
       'precision_mean', 'precision_std', 'recall_mean', 'recall_std',
       'f1_mean', 'f1_std', 'roc_auc_mean', 'roc_auc_std', 'pr_auc_mean',
       'pr_auc_std', 'accuracy_mean', 'accuracy_std', 'feature_set_raw',
       'features_raw', 'feature_set_alias', 'features_alias', 'feature_count'],
      dtype='str')


## 12. Interpretation Notes

해석은 아래 기준으로 작성한다.

- `3C1` 결과는 세 행동 지표 중 어떤 단일 feature가 가장 강한지 보여준다.
- `3C2` 결과가 `3C1`보다 좋아지면 두 행동 지표가 보완적인 신호를 가진다고 해석할 수 있다.
- `3C3` 결과가 가장 좋으면 세 행동 지표를 모두 쓰는 것이 유리하다는 뜻이다.
- `base_beh_*`가 대응되는 `beh_*`보다 좋아지면 total clicks와 active days가 추가 설명력을 가진다.
- `base_beh_*` 개선폭이 작으면 최소 행동 feature만으로도 base feature의 주요 신호를 상당 부분 대체한다고 볼 수 있다.
- F1이 높아도 recall이 낮으면 at-risk 학생을 많이 놓치는 모델일 수 있다.
- precision이 낮으면 너무 많은 학생을 at-risk로 flag하는 모델일 수 있다.
- `week5 -> week7 -> week10`으로 갈수록 성능이 좋아진다면, 시간이 지나며 행동 신호가 더 뚜렷해진다는 해석이 가능하다.

이 실험은 최소 feature 실험이다. 성능이 낮더라도 feature가 부족해서 생기는 한계인지, 모델 한계인지 분리해서 서술해야 한다.